In [ ]:

!pip -q install torch torchvision pandas datasets scipy matplotlib tqdm

from __future__ import annotations

import copy
import gc
import json
import math
import os
import random
import time
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from matplotlib import pyplot as plt
from scipy import stats
from torch.func import functional_call
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from tqdm.auto import tqdm

try:
    import google.colab  
    RUN_IN_COLAB = True
except Exception:
    RUN_IN_COLAB = False

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


In [ ]:
@dataclass(frozen=True)
class Config:
    dataset_id: str = "uoft-cs/cifar10"
    boundary_classes: Tuple[int, ...] = (0, 1, 2, 3, 4)
    continuation_classes: Tuple[int, ...] = (5, 6, 7, 8, 9)
    train_per_class: int = 1000
    test_per_class: int = 500
    batch_size: int = 128
    boundary_epochs: int = 12
    boundary_lr: float = 1e-3
    response_lrs: Tuple[float, ...] = (3e-4, 1e-3)
    beta1: float = 0.9
    beta2: float = 0.999
    adam_eps: float = 1e-8
    adam_v_floor: float = 1e-12
    weight_decay: float = 0.0
    horizons: Tuple[int, ...] = (1, 5, 10, 20, 40)
    memory_components: Tuple[str, ...] = ("m", "log_v", "joint")
    finite_difference_scales: Tuple[float, ...] = (1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2)
    n_directions: int = 4
    response_rank: int = 8
    seed: int = 0
    num_workers: int = 0
    run_spectrum: bool = True
    force: bool = False

CFG = Config(
    seed=int(os.environ.get("ADAM_RTTP_SEED", "0")),
    force=os.environ.get("ADAM_RTTP_FORCE", "0") == "1",
)

USE_GOOGLE_DRIVE = os.environ.get("ADAM_RTTP_USE_GOOGLE_DRIVE", "1") != "0"
DRIVE_BASE = Path(os.environ.get("ADAM_RTTP_DRIVE_BASE", "/content/drive/MyDrive"))
ROOT_NAME = os.environ.get("ADAM_RTTP_ROOT_NAME", "adam_paper_aligned_response")

if RUN_IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = DRIVE_BASE / ROOT_NAME
else:
    ROOT = Path.cwd() / ROOT_NAME

TAB = ROOT / "tables"
FIG = ROOT / "figures"
CACHE = ROOT / "cache"
CHUNKS = CACHE / "chunks"
for p in (ROOT, TAB, FIG, CACHE, CHUNKS):
    p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32

print("device:", DEVICE)
print("root:", ROOT)
print("config:", CFG)
(ROOT / "config.json").write_text(json.dumps(asdict(CFG), indent=2))


In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def format_duration(seconds: float) -> str:
    seconds = max(0, int(round(float(seconds))))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h}h {m:02d}m {s:02d}s"
    if m:
        return f"{m}m {s:02d}s"
    return f"{s}s"


class EtaTimer:
    def __init__(self, total: int, label: str):
        self.total = max(int(total), 1)
        self.label = label
        self.start = time.perf_counter()
        self.done = 0

    def update(self, n: int = 1, note: str = ""):
        self.done += int(n)
        elapsed = time.perf_counter() - self.start
        rate = self.done / elapsed if elapsed > 0 else float("nan")
        remaining = (self.total - self.done) / rate if np.isfinite(rate) and rate > 0 else float("nan")
        eta = format_duration(remaining) if np.isfinite(remaining) else "unknown"
        suffix = f" | {note}" if note else ""
        print(f"[{self.label}] {self.done}/{self.total} | elapsed {format_duration(elapsed)} | ETA {eta}{suffix}")


def atomic_write_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)


def atomic_torch_save(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    tmp.replace(path)


seed_everything(CFG.seed)


COMPONENT_SEED = {"m": 101, "log_v": 211, "joint": 307}


In [ ]:
class HFCifarTask(Dataset):
    def __init__(self, hf_split, classes: Sequence[int], per_class: int, train: bool):
        self.classes = tuple(int(c) for c in classes)
        self.label_map = {c: i for i, c in enumerate(self.classes)}
        counts = {c: 0 for c in self.classes}
        rows = []
        for row in hf_split:
            y = int(row["label"])
            if y in counts and counts[y] < per_class:
                rows.append(row)
                counts[y] += 1
            if all(v >= per_class for v in counts.values()):
                break
        missing = {c: per_class - n for c, n in counts.items() if n < per_class}
        if missing:
            raise ValueError(f"not enough examples per class: {missing}")
        self.rows = rows
        self.transform = (
            v2.Compose([
                v2.ToImage(),
                v2.RandomCrop(32, padding=4),
                v2.RandomHorizontalFlip(),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
            ])
            if train else
            v2.Compose([
                v2.ToImage(),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
            ])
        )

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        return self.transform(row["img"]), self.label_map[int(row["label"])]


def make_data(seed: int):
    seed_everything(seed)
    ds = load_dataset(CFG.dataset_id)
    A_tr = HFCifarTask(ds["train"], CFG.boundary_classes, CFG.train_per_class, True)
    B_tr = HFCifarTask(ds["train"], CFG.continuation_classes, CFG.train_per_class, True)
    A_te = HFCifarTask(ds["test"], CFG.boundary_classes, CFG.test_per_class, False)
    B_te = HFCifarTask(ds["test"], CFG.continuation_classes, CFG.test_per_class, False)
    kw = dict(batch_size=CFG.batch_size, num_workers=CFG.num_workers, pin_memory=torch.cuda.is_available())
    return {
        "A_train": DataLoader(A_tr, shuffle=True, generator=torch.Generator().manual_seed(seed), **kw),
        "B_train": DataLoader(B_tr, shuffle=True, generator=torch.Generator().manual_seed(seed + 10_000), **kw),
        "A_test": DataLoader(A_te, shuffle=False, **kw),
        "B_test": DataLoader(B_te, shuffle=False, **kw),
    }


def materialize_batches(loader: DataLoader, H: int):
    out = []
    it = iter(loader)
    while len(out) < H:
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(loader)
            x, y = next(it)
        out.append((x.to(device=DEVICE, dtype=DTYPE), y.to(DEVICE)))
    return out


data = make_data(CFG.seed)
future_batches = materialize_batches(data["B_train"], max(CFG.horizons))
print("Task A classes:", CFG.boundary_classes)
print("Task B classes:", CFG.continuation_classes)
print("future batches:", len(future_batches))


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(128, n_classes)

    def forward(self, x):
        return self.fc(self.features(x).flatten(1))


model_template = SmallCNN(n_classes=5).to(device=DEVICE, dtype=DTYPE)
param_items = list(model_template.named_parameters())
param_names = [name for name, _ in param_items]
param_shapes = [(p.shape, p.numel()) for _, p in param_items]
NUM_PARAMS = int(sum(n for _, n in param_shapes))


def flatten_model(model: nn.Module) -> torch.Tensor:
    return torch.cat([p.detach().reshape(-1) for p in model.parameters()])


def vector_to_param_dict(theta: torch.Tensor) -> Dict[str, torch.Tensor]:
    out = {}
    offset = 0
    for name, (shape, n) in zip(param_names, param_shapes):
        out[name] = theta[offset:offset+n].view(shape)
        offset += n
    assert offset == theta.numel()
    return out


def batch_loss(theta: torch.Tensor, xb: torch.Tensor, yb: torch.Tensor) -> torch.Tensor:
    logits = functional_call(model_template, vector_to_param_dict(theta), (xb,))
    return F.cross_entropy(logits, yb)


print("SmallCNN parameters:", NUM_PARAMS)


In [ ]:
@dataclass
class AdamState:
    theta: torch.Tensor
    m: torch.Tensor
    v: torch.Tensor
    step: int


@dataclass
class AdamTangent:
    dtheta: torch.Tensor
    dm: torch.Tensor
    dv: torch.Tensor


def clone_adam_state(s: AdamState) -> AdamState:
    return AdamState(s.theta.detach().clone(), s.m.detach().clone(), s.v.detach().clone(), int(s.step))


def adam_denominator_and_dsqrt(v_hat: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    floor = CFG.adam_eps ** 2
    active = v_hat > floor
    sqrt_v = torch.sqrt(torch.clamp(v_hat, min=floor))
    dsqrt_dv = torch.where(active, 0.5 / sqrt_v, torch.zeros_like(v_hat))
    return sqrt_v + CFG.adam_eps, dsqrt_dv


def grad_only(theta: torch.Tensor, xb: torch.Tensor, yb: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    theta_req = theta.detach().requires_grad_(True)
    loss = batch_loss(theta_req, xb, yb)
    g = torch.autograd.grad(loss, theta_req)[0]
    return loss.detach(), g.detach()


def grad_and_hvp(theta: torch.Tensor, vec: torch.Tensor, xb: torch.Tensor, yb: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    theta_req = theta.detach().requires_grad_(True)
    loss = batch_loss(theta_req, xb, yb)
    g = torch.autograd.grad(loss, theta_req, create_graph=True)[0]
    hvp = torch.autograd.grad(torch.dot(g, vec), theta_req, retain_graph=False)[0]
    return g.detach(), hvp.detach()


def adam_step(state: AdamState, xb: torch.Tensor, yb: torch.Tensor, lr: float) -> Tuple[AdamState, float]:
    loss, g = grad_only(state.theta, xb, yb)
    step = state.step + 1
    b1, b2 = CFG.beta1, CFG.beta2
    m_next = b1 * state.m + (1 - b1) * g
    v_next = b2 * state.v + (1 - b2) * g.square()
    c1 = 1 - b1 ** step
    c2 = 1 - b2 ** step
    m_hat = m_next / c1
    v_hat = v_next / c2
    denom, _ = adam_denominator_and_dsqrt(v_hat)
    theta_next = state.theta - lr * m_hat / denom
    return AdamState(theta_next.detach(), m_next.detach(), v_next.detach(), step), float(loss)


def adam_tangent_step(
    nominal: AdamState,
    tangent: AdamTangent,
    xb: torch.Tensor,
    yb: torch.Tensor,
    lr: float,
    restricted_no_dv: bool = False,
) -> Tuple[AdamState, AdamTangent]:
    g, h_dtheta = grad_and_hvp(nominal.theta, tangent.dtheta, xb, yb)
    step = nominal.step + 1
    b1, b2 = CFG.beta1, CFG.beta2

    m_next = b1 * nominal.m + (1 - b1) * g
    v_next = b2 * nominal.v + (1 - b2) * g.square()

    dm_next = b1 * tangent.dm + (1 - b1) * h_dtheta
    if restricted_no_dv:
        dv_next = torch.zeros_like(tangent.dv)
    else:
        dv_next = b2 * tangent.dv + 2 * (1 - b2) * g * h_dtheta

    c1 = 1 - b1 ** step
    c2 = 1 - b2 ** step
    m_hat = m_next / c1
    v_hat = v_next / c2
    dm_hat = dm_next / c1
    dv_hat = dv_next / c2
    denom, dsqrt_dv = adam_denominator_and_dsqrt(v_hat)

    d_update = dm_hat / denom - m_hat * dsqrt_dv * dv_hat / denom.square()
    dtheta_next = tangent.dtheta - lr * d_update
    theta_next = nominal.theta - lr * m_hat / denom

    if not torch.isfinite(dtheta_next).all():
        raise FloatingPointError("non-finite Adam tangent")

    return (
        AdamState(theta_next.detach(), m_next.detach(), v_next.detach(), step),
        AdamTangent(dtheta_next.detach(), dm_next.detach(), dv_next.detach()),
    )


In [ ]:
def train_or_load_boundary() -> Tuple[AdamState, pd.DataFrame]:
    path = CACHE / f"boundary_adam_smallcnn_seed{CFG.seed}.pt"
    if path.exists() and not CFG.force:
        obj = torch.load(path, map_location=DEVICE)
        state = AdamState(obj["theta"].to(DEVICE), obj["m"].to(DEVICE), obj["v"].to(DEVICE), int(obj["step"]))
        losses = pd.DataFrame(obj.get("losses", []))
        print("loaded boundary:", path)
        return state, losses

    seed_everything(CFG.seed)
    theta0 = flatten_model(model_template).to(device=DEVICE, dtype=DTYPE)
    state = AdamState(theta0, torch.zeros_like(theta0), torch.zeros_like(theta0), 0)
    rows = []
    timer = EtaTimer(CFG.boundary_epochs * len(data["A_train"]), "boundary Adam Task A")
    for epoch in range(CFG.boundary_epochs):
        for xb, yb in data["A_train"]:
            xb = xb.to(device=DEVICE, dtype=DTYPE)
            yb = yb.to(DEVICE)
            state, loss = adam_step(state, xb, yb, CFG.boundary_lr)
            rows.append(dict(epoch=epoch, step=state.step, loss=loss))
            if state.step % 50 == 0:
                timer.update(n=50, note=f"step={state.step} loss={loss:.3f}")
    losses = pd.DataFrame(rows)
    atomic_torch_save({
        "theta": state.theta.detach().cpu(),
        "m": state.m.detach().cpu(),
        "v": state.v.detach().cpu(),
        "step": int(state.step),
        "losses": rows,
        "config": asdict(CFG),
    }, path)
    atomic_write_csv(losses, TAB / "boundary_losses.csv")
    print("saved boundary:", path)
    return state, losses


boundary, boundary_losses = train_or_load_boundary()
print("boundary step:", boundary.step)
print("||theta||:", float(boundary.theta.norm()))
print("||m||:", float(boundary.m.norm()))
print("||v||:", float(boundary.v.norm()))
print("min v:", float(boundary.v.min()), "zero v count:", int((boundary.v == 0).sum()))


In [ ]:
def nominal_adam_state(boundary: AdamState) -> AdamState:
    v_base = torch.clamp(boundary.v.detach(), min=CFG.adam_v_floor)
    return AdamState(boundary.theta.detach().clone(), boundary.m.detach().clone(), v_base.clone(), int(boundary.step))


def split_joint(coord: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    return coord[:NUM_PARAMS], coord[NUM_PARAMS:]


def component_dim(component: str) -> int:
    return 2 * NUM_PARAMS if component == "joint" else NUM_PARAMS


def component_radius_base(component: str, base: AdamState) -> float:
    if component == "m":
        return max(float(base.m.norm()), 1e-12)
    return 1.0


def coordinate_tangent(base: AdamState, component: str, coord: torch.Tensor) -> AdamTangent:
    z = torch.zeros(NUM_PARAMS, device=DEVICE, dtype=DTYPE)
    if component == "m":
        return AdamTangent(z.clone(), coord.clone(), z.clone())
    if component == "log_v":
        return AdamTangent(z.clone(), z.clone(), base.v * coord)
    if component == "joint":
        zm, zlog = split_joint(coord)
        return AdamTangent(z.clone(), zm.clone(), base.v * zlog)
    raise ValueError(component)


def state_from_coordinate(base: AdamState, component: str, coord: torch.Tensor, eps_scale: float) -> AdamState:
    if component == "m":
        return AdamState(base.theta.clone(), base.m + eps_scale * coord, base.v.clone(), base.step)
    if component == "log_v":
        return AdamState(base.theta.clone(), base.m.clone(), base.v * torch.exp(torch.clamp(eps_scale * coord, min=-20, max=20)), base.step)
    if component == "joint":
        zm, zlog = split_joint(coord)
        return AdamState(base.theta.clone(), base.m + eps_scale * zm, base.v * torch.exp(torch.clamp(eps_scale * zlog, min=-20, max=20)), base.step)
    raise ValueError(component)


def random_unit_direction(dim: int, seed: int) -> torch.Tensor:
    gen = torch.Generator(device="cpu").manual_seed(int(seed))
    z = torch.randn(dim, generator=gen, dtype=DTYPE).to(DEVICE)
    return z / z.norm().clamp_min(1e-30)


def orthonormal_directions(component: str, rank: int, seed: int, base: AdamState) -> List[torch.Tensor]:
    dim = component_dim(component)
    candidates = []
    if component == "m" and float(base.m.norm()) > 1e-12:
        candidates.append(base.m / base.m.norm().clamp_min(1e-30))
    if component == "log_v":
        candidates.append(torch.ones(dim, device=DEVICE, dtype=DTYPE) / math.sqrt(dim))
    if component == "joint":
        first = torch.zeros(dim, device=DEVICE, dtype=DTYPE)
        if float(base.m.norm()) > 1e-12:
            first[:NUM_PARAMS] = base.m / base.m.norm().clamp_min(1e-30)
        first[NUM_PARAMS:] = 1 / math.sqrt(NUM_PARAMS)
        candidates.append(first / first.norm().clamp_min(1e-30))

    vecs = []
    for cand in candidates:
        v = cand.clone()
        for q in vecs:
            v = v - torch.dot(v, q) * q
        n = v.norm()
        if float(n) > 1e-8:
            vecs.append(v / n)
    k = 0
    while len(vecs) < rank:
        v = random_unit_direction(dim, seed + 9173 + k)
        k += 1
        for q in vecs:
            v = v - torch.dot(v, q) * q
        n = v.norm()
        if float(n) > 1e-8:
            vecs.append(v / n)
    return vecs[:rank]


base_state = nominal_adam_state(boundary)
print("nominal step offset:", base_state.step)
print("nominal min v:", float(base_state.v.min()))


In [ ]:
def theta_snapshots_from_state(initial: AdamState, lr: float, horizons: Sequence[int]) -> Dict[int, torch.Tensor]:
    wanted = set(int(h) for h in horizons)
    state = clone_adam_state(initial)
    out = {}
    for local_step, (xb, yb) in enumerate(future_batches, start=1):
        state, _ = adam_step(state, xb, yb, lr)
        if local_step in wanted:
            out[local_step] = state.theta.detach().clone()
    return out


def tangent_snapshots(component: str, coord: torch.Tensor, lr: float, horizons: Sequence[int], restricted_no_dv: bool = False) -> Tuple[Dict[int, torch.Tensor], Dict[int, float]]:
    wanted = set(int(h) for h in horizons)
    nominal = clone_adam_state(base_state)
    tangent = coordinate_tangent(base_state, component, coord)
    theta_out = {}
    dv_norm = {0: float(tangent.dv.norm())}
    for local_step, (xb, yb) in enumerate(future_batches, start=1):
        nominal, tangent = adam_tangent_step(nominal, tangent, xb, yb, lr, restricted_no_dv=restricted_no_dv)
        dv_norm[local_step] = float(tangent.dv.norm())
        if local_step in wanted:
            theta_out[local_step] = tangent.dtheta.detach().clone()
    return theta_out, dv_norm


def rel_error(pred: torch.Tensor, true: torch.Tensor, tol: float = 1e-12) -> float:
    den = float(true.norm())
    if den < tol:
        return np.nan
    return float((pred - true).norm() / den)


def cosine(pred: torch.Tensor, true: torch.Tensor, tol: float = 1e-12) -> float:
    pn = float(pred.norm())
    tn = float(true.norm())
    if min(pn, tn) < tol:
        return np.nan
    return float(torch.dot(pred, true) / (pred.norm() * true.norm()))


def run_direction(component: str, direction_id: int, lr: float) -> Tuple[pd.DataFrame, pd.DataFrame]:
    radius = component_radius_base(component, base_state)
    unit = random_unit_direction(component_dim(component), CFG.seed + 100_000 + 1000 * direction_id + COMPONENT_SEED[component])
    coord = radius * unit

    full_tangent, full_dv = tangent_snapshots(component, coord, lr, CFG.horizons, restricted_no_dv=False)
    restricted_tangent, restricted_dv = ({}, {})
    if component == "m":
        restricted_tangent, restricted_dv = tangent_snapshots(component, coord, lr, CFG.horizons, restricted_no_dv=True)

    nominal = theta_snapshots_from_state(base_state, lr, CFG.horizons)
    rows = []
    for scale in CFG.finite_difference_scales:
        plus_state = state_from_coordinate(base_state, component, coord, float(scale))
        minus_state = state_from_coordinate(base_state, component, coord, -float(scale))
        plus = theta_snapshots_from_state(plus_state, lr, CFG.horizons)
        minus = theta_snapshots_from_state(minus_state, lr, CFG.horizons)
        v_plus_nonnegative = bool(torch.isfinite(plus_state.v).all() and (plus_state.v >= 0).all())
        v_minus_nonnegative = bool(torch.isfinite(minus_state.v).all() and (minus_state.v >= 0).all())

        for H in CFG.horizons:
            true_endpoint = plus[H] - nominal[H]
            centered = (plus[H] - minus[H]) / (2 * float(scale))
            for tangent_model, tangent_by_H in [("full_coupled", full_tangent), ("restricted_no_dv", restricted_tangent)]:
                if H not in tangent_by_H:
                    continue
                tan = tangent_by_H[H]
                pred_endpoint = float(scale) * tan
                rows.append(dict(
                    optimizer="adam",
                    architecture="smallcnn",
                    seed=CFG.seed,
                    direction=direction_id,
                    learning_rate=lr,
                    beta1=CFG.beta1,
                    beta2=CFG.beta2,
                    adam_eps=CFG.adam_eps,
                    memory_component=component,
                    tangent_model=tangent_model,
                    state_parameterization="additive" if component == "m" else "log" if component == "log_v" else "additive+log",
                    nominal_memory="inherited",
                    step_offset=base_state.step,
                    horizon=H,
                    radius_scale=float(scale),
                    radius_base=radius,
                    coordinate_norm=float(coord.norm()),
                    tangent_norm=float(tan.norm()),
                    true_endpoint_norm=float(true_endpoint.norm()),
                    centered_fd_norm=float(centered.norm()),
                    endpoint_error=rel_error(pred_endpoint, true_endpoint),
                    endpoint_cosine=cosine(pred_endpoint, true_endpoint),
                    centered_error=rel_error(tan, centered),
                    centered_cosine=cosine(tan, centered),
                    v_plus_nonnegative=v_plus_nonnegative,
                    v_minus_nonnegative=v_minus_nonnegative,
                ))
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    dv_rows = []
    for tangent_model, dv_hist in [("full_coupled", full_dv), ("restricted_no_dv", restricted_dv)]:
        if not dv_hist:
            continue
        for step, value in dv_hist.items():
            dv_rows.append(dict(
                optimizer="adam",
                architecture="smallcnn",
                seed=CFG.seed,
                direction=direction_id,
                learning_rate=lr,
                memory_component=component,
                tangent_model=tangent_model,
                step=step,
                dv_norm=value,
            ))
    return pd.DataFrame(rows), pd.DataFrame(dv_rows)


In [ ]:
def run_one_step_audit() -> pd.DataFrame:
    path = TAB / "adam_one_step_audit.csv"
    if path.exists() and not CFG.force:
        return pd.read_csv(path)
    rows = []
    xb, yb = future_batches[0]
    for component in CFG.memory_components:
        radius = component_radius_base(component, base_state)
        unit = random_unit_direction(component_dim(component), CFG.seed + 2222 + COMPONENT_SEED[component])
        coord = radius * unit
        tangent = coordinate_tangent(base_state, component, coord)
        _, tangent_next = adam_tangent_step(clone_adam_state(base_state), tangent, xb, yb, CFG.response_lrs[0], restricted_no_dv=False)
        for scale in CFG.finite_difference_scales:
            plus, _ = adam_step(state_from_coordinate(base_state, component, coord, float(scale)), xb, yb, CFG.response_lrs[0])
            minus, _ = adam_step(state_from_coordinate(base_state, component, coord, -float(scale)), xb, yb, CFG.response_lrs[0])
            centered = (plus.theta - minus.theta) / (2 * float(scale))
            rows.append(dict(
                optimizer="adam",
                architecture="smallcnn",
                seed=CFG.seed,
                memory_component=component,
                learning_rate=CFG.response_lrs[0],
                radius_scale=float(scale),
                centered_error=rel_error(tangent_next.dtheta, centered),
                centered_cosine=cosine(tangent_next.dtheta, centered),
                tangent_norm=float(tangent_next.dtheta.norm()),
                centered_fd_norm=float(centered.norm()),
            ))
    out = pd.DataFrame(rows)
    atomic_write_csv(out, path)
    return out


one_step_audit = run_one_step_audit()
one_step_audit.head()


In [ ]:
def response_chunk_path(lr: float, component: str, direction: int) -> Path:
    return CHUNKS / f"response_adam_smallcnn_lr={lr:g}_mem={component}_dir={direction}.csv"


def dv_chunk_path(lr: float, component: str, direction: int) -> Path:
    return CHUNKS / f"dv_adam_smallcnn_lr={lr:g}_mem={component}_dir={direction}.csv"


all_rows = []
all_dv = []
total = len(CFG.response_lrs) * len(CFG.memory_components) * CFG.n_directions
timer = EtaTimer(total, "Adam response grid")
for lr in CFG.response_lrs:
    for component in CFG.memory_components:
        for direction in range(CFG.n_directions):
            rpath = response_chunk_path(lr, component, direction)
            dpath = dv_chunk_path(lr, component, direction)
            if rpath.exists() and dpath.exists() and not CFG.force:
                rows = pd.read_csv(rpath)
                dv_rows = pd.read_csv(dpath)
                note = "loaded"
            else:
                rows, dv_rows = run_direction(component, direction, lr)
                atomic_write_csv(rows, rpath)
                atomic_write_csv(dv_rows, dpath)
                note = "computed"
            all_rows.append(rows)
            all_dv.append(dv_rows)
            timer.update(note=f"{note} lr={lr:g} mem={component} dir={direction}")

response_raw = pd.concat(all_rows, ignore_index=True)
dv_response = pd.concat(all_dv, ignore_index=True)
atomic_write_csv(response_raw, TAB / "adam_response_raw.csv")
atomic_write_csv(dv_response, TAB / "adam_dv_response.csv")
response_raw.head()


In [ ]:
def spectrum_from_columns(columns: List[torch.Tensor]) -> Dict[str, float]:
    r = len(columns)
    G = torch.empty((r, r), dtype=torch.float64)
    for i in range(r):
        for j in range(r):
            G[i, j] = float(torch.dot(columns[i].reshape(-1), columns[j].reshape(-1)))
    G = (G + G.T) / 2
    eig = torch.linalg.eigvalsh(G).clamp_min(0)
    sing = torch.sqrt(eig).flip(0).cpu().numpy()
    energy = sing ** 2
    total = float(energy.sum())
    probs = energy / total if total > 0 else np.zeros_like(energy)
    entropy = float(-(probs[probs > 0] * np.log(probs[probs > 0])).sum()) if total > 0 else np.nan
    cum = np.cumsum(probs) if total > 0 else np.zeros_like(probs)
    row = dict(
        response_rank=r,
        frobenius_norm=math.sqrt(total),
        spectral_norm=float(sing[0]) if len(sing) else np.nan,
        stable_rank=float(total / (sing[0] ** 2)) if len(sing) and sing[0] > 0 else np.nan,
        effective_rank=math.exp(entropy) if np.isfinite(entropy) else np.nan,
        energy_top1=float(probs[0]) if len(probs) else np.nan,
        energy_top2=float(probs[:2].sum()) if len(probs) else np.nan,
        rank90=int(np.searchsorted(cum, 0.9) + 1) if total > 0 else np.nan,
    )
    for k, s in enumerate(sing, start=1):
        row[f"singular_value_{k}"] = float(s)
    return row


def spectrum_chunk_path(lr: float, component: str) -> Path:
    return CHUNKS / f"spectrum_adam_smallcnn_lr={lr:g}_mem={component}.csv"


def run_spectrum_component(lr: float, component: str) -> pd.DataFrame:
    path = spectrum_chunk_path(lr, component)
    if path.exists() and not CFG.force:
        return pd.read_csv(path)
    dirs = orthonormal_directions(component, CFG.response_rank, CFG.seed + 4444 + COMPONENT_SEED[component], base_state)
    radius = component_radius_base(component, base_state)
    rows = []
    columns_by_H = {H: [] for H in CFG.horizons}
    for j, unit in enumerate(tqdm(dirs, desc=f"spectrum {component} lr={lr:g}")):
        coord = radius * unit
        snaps, _ = tangent_snapshots(component, coord, lr, CFG.horizons, restricted_no_dv=False)
        for H in CFG.horizons:
            columns_by_H[H].append(snaps[H])
    for H in CFG.horizons:
        row = spectrum_from_columns(columns_by_H[H])
        row.update(dict(
            optimizer="adam",
            architecture="smallcnn",
            seed=CFG.seed,
            learning_rate=lr,
            beta1=CFG.beta1,
            beta2=CFG.beta2,
            memory_component=component,
            state_parameterization="additive" if component == "m" else "log" if component == "log_v" else "additive+log",
            nominal_memory="inherited",
            step_offset=base_state.step,
            horizon=H,
            radius_base=radius,
        ))
        rows.append(row)
    out = pd.DataFrame(rows)
    atomic_write_csv(out, path)
    return out


if CFG.run_spectrum:
    spectra = []
    for lr in CFG.response_lrs:
        for component in CFG.memory_components:
            spectra.append(run_spectrum_component(lr, component))
    response_spectrum = pd.concat(spectra, ignore_index=True)
else:
    response_spectrum = pd.DataFrame()
atomic_write_csv(response_spectrum, TAB / "adam_response_spectrum.csv")
response_spectrum.head()


In [ ]:
def ci95(x):
    x = pd.Series(x).dropna().to_numpy(float)
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    m = float(x.mean())
    if len(x) == 1:
        return m, np.nan, np.nan
    se = float(x.std(ddof=1) / math.sqrt(len(x)))
    q = float(stats.t.ppf(0.975, len(x)-1))
    return m, m - q * se, m + q * se


summary_rows = []
for keys, g in response_raw.groupby(["learning_rate", "memory_component", "tangent_model", "horizon", "radius_scale"]):
    lr, component, tangent_model, H, scale = keys
    row = dict(learning_rate=lr, memory_component=component, tangent_model=tangent_model, horizon=H, radius_scale=scale)
    for col in ["endpoint_error", "centered_error", "endpoint_cosine", "centered_cosine", "tangent_norm", "true_endpoint_norm", "centered_fd_norm"]:
        m, lo, hi = ci95(g[col])
        row[col] = m
        row[col + "_lo"] = lo
        row[col + "_hi"] = hi
    row["n"] = len(g)
    summary_rows.append(row)
response_summary_by_scale = pd.DataFrame(summary_rows)
atomic_write_csv(response_summary_by_scale, TAB / "adam_response_summary_by_scale.csv")

best_rows = []
for keys, g in response_summary_by_scale.groupby(["learning_rate", "memory_component", "tangent_model", "horizon"]):
    finite = g[np.isfinite(g.centered_error)].copy()
    if len(finite):
        best = finite.loc[finite.centered_error.idxmin()].to_dict()
        best["selection"] = "min_centered_error_scale"
        best_rows.append(best)
adam_best_scale_summary = pd.DataFrame(best_rows)
atomic_write_csv(adam_best_scale_summary, TAB / "adam_best_scale_summary.csv")

quality = []
def add_check(check, passed, detail, blocking=True):
    quality.append(dict(check=check, passed=bool(passed), blocking=bool(blocking), detail=str(detail)))

add_check("one_step_audit_finite", bool(np.isfinite(one_step_audit.centered_error).all()), one_step_audit.centered_error.describe().to_dict())
add_check("response_raw_finite", bool(np.isfinite(response_raw[["endpoint_error", "centered_error", "endpoint_cosine", "centered_cosine"]].to_numpy()).all()), response_raw.isna().sum().to_dict())
add_check("v_perturbations_nonnegative", bool(response_raw.v_plus_nonnegative.all() and response_raw.v_minus_nonnegative.all()), "all plus/minus v states nonnegative")

for (lr, comp), g in adam_best_scale_summary[(adam_best_scale_summary.tangent_model == "full_coupled")].groupby(["learning_rate", "memory_component"]):
    h20 = g[g.horizon == 20]
    if len(h20):
        err = float(h20.centered_error.iloc[0])
        add_check(f"local_centered_error_lr={lr:g}_mem={comp}_H20", err < 0.25, f"best centered error at H=20 is {err:.3g}", blocking=False)

if "restricted_no_dv" in response_raw.tangent_model.unique():
    m_full = adam_best_scale_summary[(adam_best_scale_summary.memory_component == "m") & (adam_best_scale_summary.tangent_model == "full_coupled")]
    m_rest = adam_best_scale_summary[(adam_best_scale_summary.memory_component == "m") & (adam_best_scale_summary.tangent_model == "restricted_no_dv")]
    merged = m_full.merge(m_rest, on=["learning_rate", "memory_component", "horizon"], suffixes=("_full", "_restricted"))
    if len(merged):
        diff = (merged.centered_error_restricted - merged.centered_error_full).abs().mean()
        add_check("full_vs_restricted_m_difference_observed", diff > 1e-3, f"mean abs centered-error gap is {diff:.3g}", blocking=False)

quality_report = pd.DataFrame(quality)
atomic_write_csv(quality_report, TAB / "adam_quality_report.csv")
quality_report


In [ ]:
def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG / f"{name}.png", dpi=220, bbox_inches="tight")
    plt.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.show()
    plt.close()

plot_df = response_summary_by_scale[(response_summary_by_scale.tangent_model == "full_coupled") & (response_summary_by_scale.horizon == 20)]
plt.figure(figsize=(7, 4.5))
for (lr, comp), g in plot_df.groupby(["learning_rate", "memory_component"]):
    g = g.sort_values("radius_scale")
    plt.loglog(g.radius_scale, g.centered_error, marker="o", label=f"{comp}, lr={lr:g}")
plt.xlabel("finite-difference radius scale")
plt.ylabel("centered derivative relative error")
plt.title("Adam full coupled response locality, H=20")
plt.legend(frameon=False)
savefig("adam_full_coupled_locality_H20")

m_df = response_summary_by_scale[response_summary_by_scale.memory_component == "m"]
plt.figure(figsize=(7, 4.5))
for (lr, model_name), g in m_df[m_df.horizon == 20].groupby(["learning_rate", "tangent_model"]):
    g = g.sort_values("radius_scale")
    plt.loglog(g.radius_scale, g.centered_error, marker="o", label=f"{model_name}, lr={lr:g}")
plt.xlabel("finite-difference radius scale")
plt.ylabel("centered derivative relative error")
plt.title("Adam m-response: full coupled vs restricted, H=20")
plt.legend(frameon=False)
savefig("adam_m_full_vs_restricted_H20")

if len(response_spectrum):
    plt.figure(figsize=(7, 4.5))
    for (lr, comp), g in response_spectrum.groupby(["learning_rate", "memory_component"]):
        g = g.sort_values("horizon")
        plt.plot(g.horizon, g.effective_rank, marker="o", label=f"{comp}, lr={lr:g}")
    plt.xlabel("horizon")
    plt.ylabel("effective rank")
    plt.title("Adam response effective rank")
    plt.legend(frameon=False)
    savefig("adam_response_effective_rank")


In [ ]:
manifest = {
    "notebook": "adam_paper_aligned_response.ipynb",
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "device": str(DEVICE),
    "torch": torch.__version__,
    "config": asdict(CFG),
    "scientific_object": "partial finite-horizon Adam optimizer-memory response blocks R_H^(m), R_H^(log_v), and joint [R_H^(m), R_H^(log_v)]",
    "adam_denominator": "sqrt(clamp(v_hat, adam_eps^2)) + adam_eps",
    "boundary_state": "inherited Adam m, v, and step counter after Task A",
    "continuation": "Task B materialized minibatch sequence held fixed for tangent and finite differences",
}
(ROOT / "manifest.json").write_text(json.dumps(manifest, indent=2))

zip_path = ROOT / "adam_paper_aligned_response_artifacts.zip"
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in (TAB, FIG):
        for path in folder.rglob("*"):
            if path.is_file():
                zf.write(path, arcname=str(path.relative_to(ROOT)))
    zf.write(ROOT / "config.json", arcname="config.json")
    zf.write(ROOT / "manifest.json", arcname="manifest.json")
print("archive:", zip_path)
